# Chess Data Visualization (Memory-Efficient)

This notebook uses memory-efficient techniques to visualize large parquet files without OOM errors.

In [ ]:
import pyarrow.parquet as pq
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import glob
import gc

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Configuration
SAMPLE_SIZE = 50000  # Rows to sample per file
MAX_FILES = 5        # Max files to sample from

## Inspect Files Without Loading (PyArrow Metadata)

In [ ]:
parquet_files = sorted(glob.glob('data/*.parquet'))
print(f"Found {len(parquet_files)} parquet files\n")

# Get metadata without loading data
total_rows = 0
file_info = []

for f in parquet_files:
    pf = pq.ParquetFile(f)
    rows = pf.metadata.num_rows
    total_rows += rows
    file_info.append({'file': Path(f).name, 'rows': rows})
    
print(f"Total rows across all files: {total_rows:,}")
print(f"\nSchema:")
print(pq.ParquetFile(parquet_files[0]).schema_arrow)

In [ ]:
# Rows per file
info_df = pd.DataFrame(file_info)
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(info_df)), info_df['rows'], color='steelblue')
ax.set_xlabel('File Index')
ax.set_ylabel('Rows')
ax.set_title(f'Rows per Parquet File (Total: {total_rows:,})')
plt.tight_layout()
plt.show()

## Load Sampled Data (Memory-Efficient)

In [ ]:
def sample_parquet(filepath, n_rows=10000):
    """Read only first n_rows from a parquet file."""
    pf = pq.ParquetFile(filepath)
    # Read first batch only
    first_batch = next(pf.iter_batches(batch_size=n_rows))
    return first_batch.to_pandas()

# Sample from a subset of files
samples = []
files_to_sample = parquet_files[:MAX_FILES]

for f in files_to_sample:
    sample = sample_parquet(f, n_rows=SAMPLE_SIZE // MAX_FILES)
    samples.append(sample)
    print(f"Sampled {len(sample):,} rows from {Path(f).name}")

df = pd.concat(samples, ignore_index=True)
del samples
gc.collect()

print(f"\nTotal sampled: {len(df):,} rows ({len(df)/total_rows*100:.2f}% of dataset)")

## Data Overview

In [ ]:
print("Columns:", df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMemory Usage:")
print(f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
df.describe()

In [ ]:
df.head(10)

## Missing Values

In [ ]:
missing_pct = (df.isnull().sum() / len(df)) * 100

fig, ax = plt.subplots(figsize=(8, 4))
missing_pct.plot(kind='bar', ax=ax, color='coral')
ax.set_ylabel('Missing %')
ax.set_title('Missing Values by Column')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Centipawn (cp) Distribution

In [ ]:
cp_valid = df['cp'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(cp_valid.clip(-1000, 1000), bins=100, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Centipawn Evaluation')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Centipawn Distribution (clipped ±1000)')
axes[0].axvline(0, color='red', linestyle='--', label='Equal')
axes[0].legend()

axes[1].boxplot(cp_valid.clip(-2000, 2000), vert=True)
axes[1].set_ylabel('Centipawn')
axes[1].set_title('Centipawn Box Plot')

plt.tight_layout()
plt.show()

## Depth Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df['depth'].value_counts().sort_index().plot(kind='bar', ax=ax, color='seagreen')
ax.set_xlabel('Search Depth')
ax.set_ylabel('Count')
ax.set_title('Engine Search Depth Distribution')
plt.tight_layout()
plt.show()

## Knodes Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['knodes'] / 1000, bins=50, color='purple', edgecolor='white', alpha=0.7)
ax.set_xlabel('Knodes (millions)')
ax.set_ylabel('Frequency')
ax.set_title('Nodes Searched Distribution')
plt.tight_layout()
plt.show()

## Mate Positions

In [ ]:
mate_positions = df[df['mate'].notna()]
print(f"Positions with mate: {len(mate_positions):,} ({len(mate_positions)/len(df)*100:.2f}%)")

if len(mate_positions) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    mate_values = mate_positions['mate'].clip(-20, 20)
    ax.hist(mate_values, bins=range(-21, 22), color='crimson', edgecolor='white')
    ax.set_xlabel('Mate in N moves')
    ax.set_ylabel('Frequency')
    ax.set_title('Mate Distance Distribution')
    ax.axvline(0, color='black', linestyle='--')
    plt.tight_layout()
    plt.show()

## Side to Move

In [ ]:
df['side'] = df['fen'].str.split().str[1]

side_counts = df['side'].value_counts()
print(side_counts)

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(side_counts, labels=['White', 'Black'], autopct='%1.1f%%',
       colors=['#EEEEEE', '#333333'], wedgeprops={'edgecolor': 'gray'})
ax.set_title('Side to Move')
plt.show()

## Evaluation by Side

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for side, color, label in [('w', 'lightgray', 'White'), ('b', 'dimgray', 'Black')]:
    subset = df[df['side'] == side]['cp'].dropna().clip(-1000, 1000)
    ax.hist(subset, bins=100, alpha=0.6, label=f"{label} to move", color=color)

ax.set_xlabel('Centipawn')
ax.set_ylabel('Frequency')
ax.set_title('Evaluation by Side to Move')
ax.axvline(0, color='red', linestyle='--')
ax.legend()
plt.tight_layout()
plt.show()

## PV Line Length

In [ ]:
df['pv_len'] = df['line'].str.split().str.len()

fig, ax = plt.subplots(figsize=(10, 5))
df['pv_len'].value_counts().sort_index().head(30).plot(kind='bar', ax=ax, color='teal')
ax.set_xlabel('PV Length')
ax.set_ylabel('Count')
ax.set_title('Principal Variation Length')
plt.tight_layout()
plt.show()

## Correlation Heatmap

In [ ]:
numeric_cols = ['depth', 'knodes', 'cp', 'mate', 'pv_len']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=ax, fmt='.2f')
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

## Sample Positions

In [ ]:
print("Sample FEN positions:")
for _, row in df.sample(5).iterrows():
    print(f"\nFEN: {row['fen']}")
    print(f"  cp={row['cp']}, mate={row['mate']}, depth={row['depth']}")

## Cleanup

In [ ]:
del df
gc.collect()
print("Memory freed.")